# M1.4_特征线与行波 — 交互式实验工作台（由 HTML 演示网页生成）

> **所属模块**：`01_M1_物理与解析基准`  
> **实验定位**：工科本科生计算物理与 SciML 交互式上机实验  
> **运行环境**：本地 Jupyter / VS Code / Colab / 个人笔记本 CPU  

---

## 本讲演示页序导览（Notebook 章节与之严格一一对应）

1. M1.4 特征线与行波
2. ① 工程场景：敲击杆的一端，另一端什么时候响应？
3. ② 先看失效：仿真里另一端“先知先觉”
4. ③ 物理直觉：信息沿特征线走，而不是瞬间填满全域
5. ④ 特征线推导：让偏导数沿路径合并
6. ⑤ d’Alembert 公式：左行波 + 右行波
7. ⑥ 反射边界：固定端反相，自由端同相
8. ⑦ 能量恒等式：传播方向之外的第二条证据
9. ⑧ 可运行代码：解析对照与特征脚点
10. ⑨ 验证三件套：精确解、收敛、守恒
11. ⑩ 从特征线到上风格式与 CFL
12. ⑪ POE：先预测，再运行，再解释
13. ⑫ 现场诊断挑战（代码面试）：看证据，找病因
14. ⑬ G1 验收与下一站

> 对齐说明：本 Notebook 由 `slides/*_演示网页.html` 自动生成；
> 代码一律从 `code/solution.py` 真实导入，函数签名由磁盘源码决定，不会漂移。


In [ ]:
import os

# ---- 单元根目录定位（不依赖任何绝对路径，换机 / Colab 均可运行）----
def find_unit_dir():
    here = globals().get("__file__")
    cands = []
    if here:
        cands.append(os.path.dirname(os.path.abspath(here)))
    cands.append(os.getcwd())
    for base in list(cands):
        d = base
        for _ in range(5):
            parent = os.path.dirname(d)
            if not parent or parent == d:
                break
            d = parent
            cands.append(d)
    for d in cands:
        if os.path.exists(os.path.join(d, "code", "solution.py")):
            return d
    for d in cands:                       # L0.1 等无 solution.py 的单元，退而求其次
        if os.path.exists(os.path.join(d, "code")) and os.path.exists(os.path.join(d, "assets")):
            return d
    return os.getcwd()

UNIT_DIR = find_unit_dir()
CODE_DIR = os.path.join(UNIT_DIR, "code")
ASSET_DIR = os.path.join(UNIT_DIR, "assets")
import sys
if UNIT_DIR not in sys.path:      # 让 cell-src 里的 `from benchmarks.xxx import ...` 可用
    sys.path.insert(0, UNIT_DIR)
print("单元根目录:", UNIT_DIR)
print("代码目录  :", CODE_DIR, "| 存在:", os.path.isdir(CODE_DIR))
print("配图目录  :", ASSET_DIR, "| 存在:", os.path.isdir(ASSET_DIR))

import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except Exception:
    pass
print("NumPy 版本:", np.__version__)

## 逐页理论与实验（与演示页序一致）

### 第 2 页 · ① 工程场景：敲击杆的一端，另一端什么时候响应？

- ① 工程场景：敲击杆的一端，另一端什么时候响应？
- 一根长杆，      t  =  0   t=0         t    =        0     时在左端施加一个短促脉冲，另一端装有传感器。工程师真正关心的不是“最终会不会动”，而是 最早 出现响应的时刻：它决定了报警阈值、采样率
- 脉冲前缘初始位于       x  =  −  0.8   x=-0.8         x    =        −  0.8     m，波速       c  =  1   c=1         c    =        1  
- 传感器位于       x  =  3.0   x=3.0         x    =        3.0     m；
- 物理上最早的到达时间是        t   p  h  y  s    =  (  3.0  −  (  −  0.8  )  )  /  1  =  3.8   t_{phys}=(3.0-(-0.8))/1=3.8          
- 30 秒课堂思考 ：既然        t   p  h  y  s    =  3.8   t_{phys}=3.8          t           p  h  ys      ​              =        3


### 第 3 页 · ② 先看失效：仿真里另一端“先知先觉”

- ② 先看失效：仿真里另一端“先知先觉”
- text
- 初值支集 = [-1.20, -0.80]，远端探测点 x = 3.00
- 物理到达时间 t_phys = 3.8000
- t = 2.00 / 3.00 / 3.70   解析解 |u(x_far)| = 0.000e+00   ← 波还没到，解严格为零
- CFL = 0.5：数值首次响应 t_num = 3.7100  (t_num/t_phys = 0.976)


### 第 4 页 · ③ 物理直觉：信息沿特征线走，而不是瞬间填满全域

- ③ 物理直觉：信息沿特征线走，而不是瞬间填满全域
- 把       x   x         x    –      t   t         t     平面画出来，扰动的传播路径就是两条直线       x  ∓  c  t  =  const   x\mp ct=\text{con
- 扰动传播
- 热方程        u  t   =  α   u   x  x     u_t=\alpha u_{xx}          u          t     ​              =        α   u         
- 波动方程        u   t  t    =   c  2    u   x  x     u_{tt}=c^2u_{xx}          u           tt      ​              =         
- 两个必须分清的术语： 依赖域 （domain of dependence）——目标点       (   x  0   ,  t  )   (x_0,t)         (   x          0     ​            


### 第 5 页 · ④ 特征线推导：让偏导数沿路径合并

- ④ 特征线推导：让偏导数沿路径合并
- 设曲线       x  =  x  (  t  )   x=x(t)         x    =        x  (  t  )    ，沿曲线对       u  (  x  (  t  )  ,  t  )   u(x(t),t
- d   d  t    u  (  x  (  t  )  ,  t  )  =   u  t   +   x  ′   (  t  )      u  x   .   \frac{d}{dt}u(x(t),t)=u_t+x'(t)\,u_
- 当        x  ′   (  t  )  =  c   x'(t)=c          x           ′          (  t  )    =        c     时，输运方程        u  t   +
- x  −  c  t  =  const  ,    u  (  x  ,  t  )  =   u  0   (  x  −  c  t  )  .   x-ct=\text{const},\qquad u(x,t)=u_0(x-ct).
- 物理含义 ：初值形状以速度       c   c         c     向右平移、不发生畸变。      c  <  0   c<0         c    <        0     时公式形式不变，但传播方向改为向左， 入流


### 第 6 页 · ⑤ d’Alembert 公式：左行波 + 右行波

- ⑤ d’Alembert 公式：左行波 + 右行波
- 整条直线上的初值问题有闭式解：
- u  (  x  ,  t  )  =    f  (  x  −  c  t  )  +  f  (  x  +  c  t  )   2   +   1   2  c     ∫   x  −  c  t    x  +  c  t  
- f  (  x  )   f(x)         f  (  x  )    ：初位移，      g  (  x  )   g(x)         g  (  x  )    ：初速度；单位分别为位移、位移/时间；
- 第一项把初位移对半劈成 右行波 与 左行波 ；
- 第二项把初速度在依赖域       [  x  −  c  t  ,     x  +  c  t  ]   [x-ct,\ x+ct]         [  x    −        c  t  ,       x    +      


### 第 7 页 · ⑥ 反射边界：固定端反相，自由端同相

- ⑥ 反射边界：固定端反相，自由端同相
- 在有限区间 $0
- 数学条件
- 反射特征
- 镜像延拓
- 固定端        u  =  0   u=0         u    =        0      反射波反相  奇延拓


### 第 8 页 · ⑦ 能量恒等式：传播方向之外的第二条证据

- ⑦ 能量恒等式：传播方向之外的第二条证据
- 定义波动能量
- E  (  t  )  =   1  2    ∫  0  L    (   u  t  2   +   c  2    u  x  2   )   d  x  .   E(t)=\frac{1}{2}\int_0^L\left(u_t^2
- 对固定端或自由端，分部积分给出       d  E  /  d  t  =  0   dE/dt=0         d  E  /  d  t    =        0    ；有阻尼或吸收边界时能量应单调下降。
- 离散侧必须用 守恒型 能量，否则会得到假漂移：
- E   n  +  1  /  2    =    Δ  x   2    [   ∑  j    w  j     (     u  j   n  +  1    −   u  j  n     Δ  t    )   2   +   c


### 第 9 页 · ⑧ 可运行代码：解析对照与特征脚点

- ⑧ 可运行代码：解析对照与特征脚点
- python · 可运行代码：解析对照与特征脚点
- 可编辑 · Shift+Enter 运行  ↺ 重置  ▶ 运行
- import  numpy  as  np
- from  benchmarks.characteristics_wave  import  (
- characteristic_feet, d_alembert, gaussian_pulse, leapfrog_wave)


### 第 10 页 · ⑨ 验证三件套：精确解、收敛、守恒

- ⑨ 验证三件套：精确解、收敛、守恒
- 本机实测（ python code/solution.py ）
- 精确解对照  d’Alembert 解析解 vs leapfrog 数值解  解析残差       8.3  ×  1   0   −  11     8.3\times10^{-11}         8.3    ×        1 
- 收敛性  固定 CFL，      N   N         N     逐次加倍        3.57  ×  1   0   −  3    →  7.32  ×  1   0   −  4    →  1.94  ×  1   0
- 守恒与能量  守恒型离散能量（梯形权重 + 鬼点边界）  固定端漂移       3.9  ×  1   0   −  15     3.9\times10^{-15}         3.9    ×        1   0      
- 只展示一张“看起来很像”的波形图不构成验证。 报告至少包含误差定义、采样时刻、CFL 与边界条件。


### 第 11 页 · ⑩ 从特征线到上风格式与 CFL

- ⑩ 从特征线到上风格式与 CFL
- 对       c  >  0   c>0         c    >        0    ，沿入流特征线取值的显式迎风差分写成
- u  j   n  +  1    =   u  j  n   −  ν   (   u  j  n   −   u   j  −  1   n   )   ,    ν  =    c  Δ  t    Δ  x    .   u_j^{
- 稳定域是       0  ≤  ν  ≤  1   0\le\nu\le1         0    ≤        ν    ≤        1    ，不是       ∣  ν  ∣  ≤  1   |\nu|\le1     
- 物理来源：一个时间步内的信息传播距离       c  Δ  t   c\Delta t         c  Δ  t     不能跨过多个网格单元（      ν  ≤  1   \nu\le1         ν    ≤      
- F2 实测证据 （ code/solution.py  第 ⑤ 节）：


### 第 12 页 · ⑪ POE：先预测，再运行，再解释

- ⑪ POE：先预测，再运行，再解释
- 运行前必须填写“我的预测”列；运行后补“观测证据”与“解释 / 失效归类”。
- 教师参考键 vs 学生工作底稿 ：下表的“观测证据”与“解释 / 失效归类”两列是 教师参考键 ，已在课前跑出并填好。学生的空白工作底稿在  notebooks/episode_lab.ipynb  第 ⑧ 节，五个改动的四列 全部留空 。
- 💡  POE 自检台 ：先独立思考（可点击填写预测），再点击右侧按钮揭晓标准解析对照。
- 👁️ 全部揭晓 / 隐藏
- 我的预测（运行前填写）


### 第 13 页 · ⑫ 现场诊断挑战（代码面试）：看证据，找病因

- ⑫ 现场诊断挑战（代码面试）：看证据，找病因
- 真实工程现场 ：一维长杆冲击响应仿真中，远端传感器在物理波前到达前产生前驱微小位移；单向应力波仿真采用迎风差分后振幅快速失稳或过度发散。请审查以下两组真实运行证据：
- 点选答案立即出解析，先看证据再找病因。全部答完给出诊断得分与排查建议。


### 第 14 页 · ⑬ G1 验收与下一站

- ⑬ G1 验收与下一站
- M1.4 是 V3 课表中 W3 的 G1 关卡节点。  G1 要求：
- 提交  benchmarks/  中的解析基准（本单元交付  benchmarks/characteristics_wave.py ）；
- 完成适定性判断题（M1.2 已建立的 F3 / F4 判别能力）；
- 完成一次“改参数先预测、再验证”的实验（本单元 POE 表）。
- 本节验收清单


## 演示页代码实验台
下面单元格直接抽取自演示网页的可运行代码（`cell-src`），按顺序执行即可复现网页上的交互结果。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from benchmarks.characteristics_wave import (
    characteristic_feet, d_alembert, gaussian_pulse, leapfrog_wave, pde_residual)

c, t = 1.0, 0.8
x0 = -1.0                                    # 脉冲中心 = 目标点
f = lambda z: gaussian_pulse(z, center=x0, width=0.2)

left, right = characteristic_feet(x0, t, c)  # 依赖域端点 = x0 -/+ c t
print(f"依赖域 = [{left:.2f}, {right:.2f}]   目标点 x0 = {x0:+.1f}")

x = np.linspace(-4.0, 4.0, 801)
print(f"解析残差 max|u_tt - c^2 u_xx| = "
      f"{pde_residual(x, 0.5, lambda xx, tt: d_alembert(xx, tt, f, c=c), c):.3e}")

for N in (801, 1600):
    xx = np.linspace(-4.0, 4.0, N)
    tt, U, dt = leapfrog_wave(xx, 1.0, c, cfl=0.9, f=f)
    err = np.max(np.abs(U[-1] - d_alembert(xx, tt[-1], f, c=c)))
    print(f"CFL = 0.9   N = {N:4d}   L-inf 误差 = {err:.3e}")

u = d_alembert(x, t, f, c=c)
plt.figure(figsize=(6.2, 3.2))
plt.plot(x, u, lw=2, color="#2563eb", label="u(x, t)  d'Alembert")
plt.axvline(left, ls="--", lw=1.3, color="#f59e0b", label=f"left foot {left:+.2f}")
plt.axvline(right, ls="--", lw=1.3, color="#10b981", label=f"right foot {right:+.2f}")
plt.axvline(x0, ls=":", lw=1.1, color="#94a3b8", label=f"target x0 = {x0:+.2f}")
plt.xlabel("x"); plt.ylabel("u(x, t)"); plt.grid(alpha=0.25); plt.legend(fontsize=7)
plt.title("t = 0.8 s : two half pulses and the characteristic feet")

## 加载本单元真实源码并核对函数签名
下面直接导入 `code/solution.py.py`（**不是副本**）；若你在磁盘上改了代码，这里的签名会立刻跟着变。

In [ ]:
import sys, inspect
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
import solution as S
SOL_PATH = os.path.join(CODE_DIR, "solution.py")
print("已从磁盘加载:", SOL_PATH)
print()
def _show():
    import ast
    tree = ast.parse(open(SOL_PATH, encoding="utf-8").read())
    for n in tree.body:
        if isinstance(n, ast.FunctionDef) and not n.name.startswith("_"):
            print("def %s" % n.name, inspect.signature(getattr(S, n.name)))
        elif isinstance(n, ast.ClassDef):
            print("class", n.name)
            for m in n.body:
                if isinstance(m, ast.FunctionDef) and (not m.name.startswith("_") or m.name == "__init__"):
                    print("    ", m.name, [a.arg for a in m.args.args])
_show()

## 运行主实验
执行下方单元格即调用本单元真实入口 `main()`，输出与演示网页一致。

In [ ]:
# 执行本单元真实入口函数（网页上的数字由此产生）
try:
    S.main()
except Exception as e:
    print("运行未成功，请把完整报错发给老师：")
    import traceback; traceback.print_exc()

## 物理场成果图展示（相对路径，换机可用）

In [ ]:
from IPython.display import Image, display
import glob
pngs = sorted(glob.glob(os.path.join(ASSET_DIR, "*.png")))
if not pngs:
    print("assets 目录下暂无 png；请先运行上面的求解入口生成图表。")
for f in pngs:
    print("展示:", os.path.basename(f))
    display(Image(filename=f))

## ① 工程场景：敲击杆的一端，另一端什么时候响应？

一根长杆，$t=0$ 时在左端施加一个短促脉冲，另一端装有传感器。工程师真正关心的是**最早**出现响应的时刻。

**30 秒课堂思考**：波速 $c = 1$ m/s，脉冲前缘初始位于 $x = -0.8$ m，探测点在 $x = 3.0$ m。那么 $t = 2$ s 时探测点应该读到什么？先把答案写下来，再运行下一节代码核对。